In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
#COND

wdc_ecm_dir = 'Specific Sulfate/WDC/ECM Data'

dataframes_dep = []
dataframes_ac_ecm = []
dataframes_dc_ecm = []

## D50 (DEP)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.D50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        skiprows=end_header_idx+1,
        header=None,
        engine="python"
    )
    dataframes_dep.append(df)

## d50 (DEP)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.d50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r",",
        skiprows=end_header_idx+2,
        header=None,
        engine="python"
    )
    dataframes_dep.append(df)

## A50 (AC-ECM)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.A50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        skiprows=end_header_idx+1,
        header=None,
        engine="python"
    )
    dataframes_ac_ecm.append(df)

## a50 (AC-ECM)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.a50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r",",
        skiprows=end_header_idx+2,
        header=None,
        engine="python"
    )
    dataframes_ac_ecm.append(df.iloc[:,0:2])

## E50 (DC-ECM)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.E50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        skiprows=end_header_idx+1,
        header=None,
        engine="python"
    )
    dataframes_dc_ecm.append(df)

## e50 (DC-ECM)
for file_path in sorted(Path(wdc_ecm_dir).glob("*.e50")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    end_header_idx = next(
        (i for i, line in enumerate(lines) if "END HEADER" in line),
        None
    )

    if end_header_idx is None:
        continue

    df = pd.read_csv(
        file_path,
        sep=r",",
        skiprows=end_header_idx+2,
        header=None,
        engine="python"
    )
    dataframes_dc_ecm.append(df.iloc[:,0:2])

full_df_dep = pd.concat(dataframes_dep, ignore_index=True) if dataframes_dep else pd.DataFrame()
full_df_ac_ecm = pd.concat(dataframes_ac_ecm, ignore_index=True) if dataframes_ac_ecm else pd.DataFrame()
full_df_dc_ecm = pd.concat(dataframes_dc_ecm, ignore_index=True) if dataframes_dc_ecm else pd.DataFrame()

full_df_dep.columns = ['Depth', 'DEP']
full_df_ac_ecm.columns = ['Depth', 'AC_ECM']
full_df_dc_ecm.columns = ['Depth', 'DC_ECM']

full_df_dep.sort_values('Depth', inplace=True)
full_df_ac_ecm.sort_values('Depth', inplace=True)
full_df_dc_ecm.sort_values('Depth', inplace=True)

# mask negative non-depth values
full_df_dep['DEP'] = full_df_dep['DEP'].mask(full_df_dep['DEP'] < 0)
full_df_ac_ecm['AC_ECM'] = full_df_ac_ecm['AC_ECM'].mask(full_df_ac_ecm['AC_ECM'] < 0)
full_df_dc_ecm['DC_ECM'] = full_df_dc_ecm['DC_ECM'].mask(full_df_dc_ecm['DC_ECM'] < 0)

full_df_dep.to_csv('lc_data_for_matchmaker/wdc/wdc_dep.txt', index=False)
full_df_ac_ecm.to_csv('lc_data_for_matchmaker/wdc/wdc_ac_ecm.txt', index=False)
full_df_dc_ecm.to_csv('lc_data_for_matchmaker/wdc/wdc_dc_ecm.txt', index=False)


In [3]:
#SULF

load1 = pd.read_excel('Specific Sulfate/WDC/WDC06A Chem Data.xls', usecols="D,H", skiprows=6, names=['Depth', 'Sulfate'])

load2 = pd.read_excel('Specific Sulfate/WDC/WDC_nssS.xlsx', sheet_name = 5,usecols="A,C", skiprows=13, names=['Depth', 'Sulfate'])

load3 = pd.read_excel('Specific Sulfate/WDC/WDC06A 577-1300 m Chemistry.xlsx', usecols="A,D", skiprows=4, names=['Depth', 'Sulfate'])

all_sulfate = pd.concat([load1, load2, load3], ignore_index=True)
all_sulfate.sort_values('Depth', inplace=True)
all_sulfate['Sulfate'] = all_sulfate['Sulfate'].mask(all_sulfate['Sulfate'] < 0)
all_sulfate.to_csv('lc_data_for_matchmaker/wdc/wdc_sulfate.txt', index=False)


In [4]:
# DRI IONS
## straight from the layer figure code

early_ion_path='Specific Sulfate/WDC/Sigl2015_SOM4_Antarctica.xlsx'

all_cols = cols_to_check = ["BC", "Na", "Sr", "nssS", "nssS_Na_ratio", "Br", "nh4", "nssCa"]
early_read_ion = pd.read_excel(early_ion_path, header=None, sheet_name='1 - WDC06A_layer_count', skiprows=1, names=["Depth_m", "Depth_mweq",	"Decimal_Year_CE",	"BC",	"Na",	"Sr",	"nssS",	"nssS_Na_ratio", "Br",	"nh4",	"nssCa"])

for col in all_cols: #remove data > -3
    early_read_ion[col]=early_read_ion[col].mask(early_read_ion[col] < -3)

early_dep_path = 'Specific Sulfate/WDC/DRI Ions.txt'
early_read_dep = pd.read_csv(early_dep_path, delimiter="\t")
early_read_dep['Cond(uS)']=early_read_dep['Cond(uS)'].mask(early_read_dep['Cond(uS)'] < 0)

name = 'shallow_bc'
save = early_read_ion[["Depth_m", "BC"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_na'
save = early_read_ion[["Depth_m", "Na"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_sr'
save = early_read_ion[["Depth_m", "Sr"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_nssS'
save = early_read_ion[["Depth_m", "nssS"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_nssS_na'
save = early_read_ion[["Depth_m", "nssS_Na_ratio"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_br'
save = early_read_ion[["Depth_m", "Br"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_nh4'
save = early_read_ion[["Depth_m", "nh4"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_nssCa'
save = early_read_ion[["Depth_m", "nssCa"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'shallow_cond'
save = early_read_dep[["Depth(m)", "Cond(uS)"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)


In [5]:
# Brittle Ion Data

ion_path='Specific Sulfate/WDC/WDC06A 577-1300 m Chemistry.xlsx'

read_ion = pd.read_excel(ion_path, header=None, skiprows=5, names=['Depth(m)', "Cl(ng/g)", "NO3(ng/g)", "SO4(ng/g)", "Na(ng/g)", "K(ng/g)", "Mg(ng/g)", "Ca(ng/g)"])

name = 'brittle_so4'
save = read_ion[["Depth(m)", "SO4(ng/g)"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'brittle_Na'
save = read_ion[["Depth(m)", "Na(ng/g)"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'brittle_so4_Na'
save = read_ion[["Depth(m)", "SO4(ng/g)", "Na(ng/g)"]].copy(deep=True)
save["SO4_Na_ratio"] = save["SO4(ng/g)"] / save["Na(ng/g)"]
save.replace([np.inf, -np.inf], np.nan, inplace=True)
save = save[["Depth(m)", "SO4_Na_ratio"]]
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'brittle_NO3'
save = read_ion[["Depth(m)", "NO3(ng/g)"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

In [6]:
# DRI 1300-2300 Data

ion_deep_path='Specific Sulfate/WDC/ion_1300_3400.csv'

read_deep_ion = pd.read_csv(ion_deep_path, sep=',', skiprows=1, usecols=[0,3,7,8,10,16,19], names=['Depth', 'BC', 'Na', 'Mg', 'S', 'Br', 'Sr'])

name = 'deep_BC'
save = read_deep_ion[["Depth", "BC"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_Na'
save = read_deep_ion[["Depth", "Na"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_Mg'
save = read_deep_ion[["Depth", "Mg"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_S'
save = read_deep_ion[["Depth", "S"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_Br'
save = read_deep_ion[["Depth", "Br"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_Sr'
save = read_deep_ion[["Depth", "Sr"]].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)

name = 'deep_NssS_Na'
div = read_deep_ion["S"] / read_deep_ion["Na"]
read_deep_ion_copy = read_deep_ion.copy(deep=True)
read_deep_ion_copy["S_Na_ratio"] = div
save = read_deep_ion_copy[["Depth", 'S_Na_ratio']].copy(deep=True)
save.iloc[:,1] = save.iloc[:,1].mask(save.iloc[:,1] < 0)
save.dropna(inplace=True)
save.to_csv(f'lc_data_for_matchmaker/wdc/wdc_{name}.txt', index=False)